# ROCm Libraries, PyTorch & AI Frameworks

<div style="background:#eef5ff; border-left:5px solid #3b6fd4; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#1e3a8a; font-size:1.05em; margin-bottom:8px;">🎯 Objectives</div>
<p>Most AI, signal-processing, and scientific workloads reduce to a few primitives — matrix multiply, convolution, FFT. To get started you will:</p><ul style='margin-bottom:0;'><li>Call <b>rocBLAS</b>, <b>MIOpen</b>, and <b>rocFFT</b> directly and measure performance</li><li>Run an end-to-end <b>PyTorch</b> training loop on an AMD GPU</li><li>Diagnose and fix common ROCm + PyTorch failures</li></ul>
</div>

## Setup — Verify the ROCm + PyTorch Environment

<div style="background:#eef5ff; border-left:5px solid #3b6fd4; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#1e3a8a; font-size:1.05em; margin-bottom:8px;">🎯 Goal</div>
Confirm Python, PyTorch, and ROCm/HIP are installed and that PyTorch can see the AMD GPU before running any exercise.
</div>

In [ ]:
import sys

print(sys.version)
print(sys.executable)

import torch

print(torch.__version__)
print(torch.version.hip)

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
# Verify ROCm PyTorch build
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"HIP available:   {torch.cuda.is_available()}")
print(f"HIP version:     {torch.version.hip}")
print(f"GPU:             {torch.cuda.get_device_name(0)}")
print(f"GPU memory:      {torch.cuda.get_device_properties(0).total_memory / (2**30):.1f} GB")

## Exercise 1 — rocBLAS GEMM

<div style="background:#eef5ff; border-left:5px solid #3b6fd4; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#1e3a8a; font-size:1.05em; margin-bottom:8px;">🎯 Goal</div>
<p><b>rocBLAS</b> is AMD's optimized BLAS implementation for ROCm GPUs. Compile and run a rocBLAS program that calls <code>rocblas_sgemm()</code> for single-precision matrix multiply, then measure throughput in GFLOPS.</p>
</div>

In [ ]:
%%writefile rocblas_gemm.cpp
#include <hip/hip_runtime.h>
#include <rocblas/rocblas.h>
#include <cstdio>
#include <cstdlib>
#include <cmath>

int main() {

    // Matrix dimensions:
    // A = M x K
    // B = K x N
    // C = M x N
    //
    // We will compute:
    //
    //     C = alpha * A * B + beta * C
    //
    // This operation is known as GEMM
    // (General Matrix Multiply).
    //
    int M = 1024, N = 1024, K = 1024;

    float alpha = 1.0f, beta = 0.1f;

    // Calculate buffer sizes in bytes
    size_t sA = M*K*sizeof(float);
    size_t sB = K*N*sizeof(float);
    size_t sC = M*N*sizeof(float);

    // Allocate host (CPU) memory
    float *h_A=(float*)malloc(sA);
    float *h_B=(float*)malloc(sB);
    float *h_C=(float*)malloc(sC);

    // Generate reproducible random input data
    srand(42);

    for(int i=0;i<M*K;i++)
        h_A[i]=(float)rand()/RAND_MAX;

    for(int i=0;i<K*N;i++)
        h_B[i]=(float)rand()/RAND_MAX;

    for(int i=0;i<K*N;i++)
        h_C[i]=(float)rand()/RAND_MAX;

    // Allocate device (GPU) memory
    float *d_A,*d_B,*d_C;

    hipMalloc(&d_A,sA);
    hipMalloc(&d_B,sB);
    hipMalloc(&d_C,sC);

    // Copy matrices from CPU memory to GPU memory
    hipMemcpy(d_A,h_A,sA,hipMemcpyHostToDevice);
    hipMemcpy(d_B,h_B,sB,hipMemcpyHostToDevice);
    hipMemcpy(d_C,h_C,sC,hipMemcpyHostToDevice);

    //
    // Create a rocBLAS context.
    //
    // Similar to creating a cuBLAS handle in CUDA.
    // The handle stores library state and execution context.
    //
    rocblas_handle handle;
    rocblas_create_handle(&handle);

    //
    // HIP events are used for accurate GPU timing.
    //
    // CPU timers are often misleading because GPU
    // operations execute asynchronously.
    //
    hipEvent_t t0,t1;
    hipEventCreate(&t0);
    hipEventCreate(&t1);

    //
    // Warm-up run.
    //
    // The first invocation may include:
    //   - ROCm runtime initialization
    //   - kernel loading/JIT compilation
    //   - cache population
    //   - library autotuning
    //
    // We exclude this overhead from measurements.
    //
    rocblas_sgemm(handle,
                  rocblas_operation_none,
                  rocblas_operation_none,
                  M, N, K,
                  &alpha,
                  d_A, M,
                  d_B, K,
                  &beta,
                  d_C, M);

    hipDeviceSynchronize();

    //
    // Benchmark phase.
    //
    // Execute GEMM 20 times and compute
    // the average execution time.
    //
    hipEventRecord(t0);

    for (int i = 0; i < 20; i++)
    {
        rocblas_sgemm(handle,
                      rocblas_operation_none,
                      rocblas_operation_none,
                      M, N, K,
                      &alpha,
                      d_A, M,
                      d_B, K,
                      &beta,
                      d_C, M);
    }

    hipEventRecord(t1);
    hipEventSynchronize(t1);

    float ms;
    hipEventElapsedTime(&ms,t0,t1);

    // Average execution time per GEMM call
    ms /= 20;

    // Copy result matrix back to host memory
    hipMemcpy(h_C, d_C, sC, hipMemcpyDeviceToHost);

    //
    // GEMM performs approximately:
    //
    //     2 * M * N * K
    //
    // floating-point operations.
    //
    // We convert that into GFLOPS
    // (billions of floating-point operations per second).
    //
    double gflops =
        2.0 * M * N * K /
        (ms * 1e-3) /
        1e9;

    printf("=== rocBLAS SGEMM ===\n");
    printf("Size:    %dx%dx%d\n", M, N, K);
    printf("Time:    %.3f ms\n", ms);
    printf("GFLOPS:  %.1f\n", gflops);

    //
    // Cleanup
    //
    rocblas_destroy_handle(handle);

    hipFree(d_A);
    hipFree(d_B);
    hipFree(d_C);

    free(h_A);
    free(h_B);
    free(h_C);

    return 0;
}

In [ ]:
import subprocess

comp = subprocess.run(['hipcc', '-O3', '-o', 'rocblas_gemm', 'rocblas_gemm.cpp',
                       '-lrocblas'], capture_output=True, text=True)
if comp.returncode != 0:
    print(f"Compile error:\n{comp.stderr}")
else:
    result = subprocess.run(['./rocblas_gemm'], capture_output=True, text=True)
    print(result.stdout)

<div style="background:#f4f6f9; border-left:5px solid #64748b; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#334155; font-size:1.05em; margin-bottom:8px;">Understanding GFLOPS</div>
<p>GEMM computes <code>C = α·A·B + β·C</code>. Each output element needs one multiply and one add per element of K, so total work is:</p><p><b>FLOPs = 2 × M × N × K</b></p><p>For M = N = K = 1024 that is about 2.15 billion operations. GFLOPS = FLOPs ÷ execution_time(s) ÷ 1e9 — it measures how efficiently the GPU runs the workload.</p>
</div>

<div style="background:#eef9f1; border-left:5px solid #2f9e6e; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#1f7a52; font-size:1.05em; margin-bottom:8px;">💡 Why GEMM matters</div>
<p>Matrix multiply is the workhorse of modern computing: neural-network training and inference, scientific simulation, and graphics all spend much of their time in GEMM. Benchmarking it measures one of the fundamental building blocks of AI hardware.</p>
</div>

## Theoretical Peak Performance and Efficiency

Measuring GFLOPS is useful, but the number alone does not tell the whole story.

To understand whether a kernel performs well, compare the measured performance with the capabilities of the GPU.

### Work Performed

For matrix multiplication (GEMM):

```text
Work = 2 × M × N × K
```

For:

```text
M = N = K = 1024
```

the total work is approximately

```text
2.15 GFLOP
```

### Interpreting Performance

Modern GPUs have finite compute throughput.
However, small GEMM problems often cannot fully utilize all available compute resources.
Measured performance depends on many factors, including:

- Problem size
- Memory hierarchy
- Library implementation
- GPU architecture

As matrix sizes increase, GPU utilization typically improves.

<div style="
background:#fff8e8;
border-left:6px solid #d8c27a;
padding:14px;
border-radius:8px;
margin:15px 0;
">

<b>Key Takeaway</b><br><br>
A lower-than-expected GFLOPS value does not necessarily indicate poor performance.
When evaluating GEMM performance, always consider:
<ul>
<li>Problem size</li>
<li>Measured GFLOPS</li>
<li>The GPU architecture</li>
</ul>
rather than comparing against a single theoretical peak number.

</div>

In [ ]:
# Analyze the rocBLAS GEMM result

M = N = K = 1024

# Total floating-point work:
# GEMM performs 2 * M * N * K floating-point operations.

flops = 2 * M * N * K

# Replace with your measured value
measured_gflops = 66202.9

print(f"Matrix size:      {M} x {N}")
print(f"Total work:       {flops/1e9:.2f} GFLOP")
print(f"Measured:         {measured_gflops/1e3:.1f} TFLOPS")
print(f"Execution time:   {flops / (measured_gflops * 1e9) * 1000:.3f} ms")

### Interpreting the Result

The measured throughput depends on many factors, including:

- GPU architecture
- Matrix size
- Memory hierarchy
- rocBLAS implementation

For this reason, the measured performance should not be compared directly with a single theoretical peak value.

Instead, compare:

- different matrix sizes;
- different GPUs;
- different library versions.

The primary goal is to understand how efficiently rocBLAS executes the GEMM operation for the selected workload.

## Exercise 2 — MIOpen Convolution with Autotune

<div style="background:#eef5ff; border-left:5px solid #3b6fd4; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#1e3a8a; font-size:1.05em; margin-bottom:8px;">🎯 Goal</div>
<p><b>MIOpen</b> is AMD's deep-learning primitive library for ROCm (the counterpart to NVIDIA's cuDNN). Run a convolution and observe how MIOpen searches for the fastest implementation on the first call, then reuses that cached choice on later calls.</p>
</div>

In [ ]:
# MIOpen convolution via PyTorch
#
# Although this example uses PyTorch, the actual convolution
# implementation is executed by the ROCm MIOpen library.
#
# PyTorch
#    ↓
# MIOpen
#    ↓
# HIP Runtime
#    ↓
# AMD GPU
#
# This allows us to observe MIOpen autotuning behavior without
# writing low-level MIOpen API code.

import torch
import torch.nn.functional as F
import time

# Verify that PyTorch can see a ROCm-compatible GPU.
if not torch.cuda.is_available():
    print("ERROR: No GPU available. Check ROCm installation.")
else:
    device = torch.device('cuda')
    print(f"GPU: {torch.cuda.get_device_name(0)}")

    #
    # Create a synthetic convolution workload similar to a
    # convolutional neural network (CNN) layer.
    #
    # Input tensor:
    #   Batch size      = 32
    #   Input channels  = 64
    #   Resolution      = 224 x 224
    #
    # Filter tensor:
    #   Output channels = 128
    #   Input channels  = 64
    #   Kernel size     = 3 x 3
    #
    x = torch.randn(32, 64, 224, 224, device=device)
    w = torch.randn(128, 64, 3, 3, device=device)

    #
    # First execution.
    #
    # This call may trigger:
    #   - ROCm runtime initialization
    #   - MIOpen algorithm search
    #   - Autotuning ("find-db" search)
    #   - Cache population
    #
    # As a result, the first run is typically slower than
    # subsequent executions.
    #
    torch.cuda.synchronize()

    t0 = time.perf_counter()

    y = F.conv2d(x, w, padding=1)

    torch.cuda.synchronize()

    first_run = (time.perf_counter() - t0) * 1000

    #
    # Benchmark phase.
    #
    # MIOpen should now have selected an optimized convolution
    # algorithm and cached the result.
    #
    # Subsequent executions typically run much faster because
    # the autotuning process is skipped.
    #
    times = []

    for _ in range(20):

        # Synchronize before timing because GPU execution
        # is asynchronous with respect to the CPU.
        torch.cuda.synchronize()

        t0 = time.perf_counter()

        y = F.conv2d(x, w, padding=1)

        torch.cuda.synchronize()

        times.append((time.perf_counter() - t0) * 1000)

    avg = sum(times) / len(times)


    # You should observe that:
    #
    #   First Run  > Cached Runs
    #
    # demonstrating MIOpen autotuning and cache reuse.
    #
    print(f"\n=== MIOpen Convolution (via PyTorch) ===")
    print(f"Input:     32×64×224×224")
    print(f"Filter:    128×64×3×3")

    print(f"First run: {first_run:.1f} ms  (includes autotune)")
    print(f"Cached:    {avg:.1f} ms  (average of 20 runs)")

    print(f"Autotune overhead: {first_run - avg:.1f} ms")

<div style="background:#fff7ea; border-left:5px solid #e0a020; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#8a5a06; font-size:1.05em; margin-bottom:8px;">⚠️ Autotuning overhead varies</div>
<p>First-run autotuning time is not fixed — it depends on the ROCm version, GPU architecture, and whether the MIOpen cache is already populated. If the cache is warm, the first call may look as fast as later ones. That is expected and demonstrates the caching mechanism.</p>
</div>